# Supplementary analysis for HRRR-SPIReS interpretation
J. Michelle Hu  
University of Utah  
June 2025

In [ ]:
import sys
import os
import copy
from pathlib import PurePath, Path

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import pandas as pd

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc
import plot_sdd_radial as psr

# Set seaborn palette
sns.set_palette('icefire')

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
basins = ['blue', 'animas', 'yampa']
verbose = True

### Radial snow depth difference plots

In [ ]:
# Built in as default values now

# # Set up directories and variables
# workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/diffs'
# script_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts'

# num_aspect_bins = 16
# palette = 'PuOr'
# outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/depth_diff'
# verbose = True

In [ ]:
# Generate radial plots of snow depth differences
def crank_snow_depth_diff_radial(basin, WY, workdir='/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/diffs',
                                 script_dir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts',
                                 outdir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/depth_diff',
                                 original=True, verbose=True, num_aspect_bins=16, palette='PuOr'):
        # Locate snow depth difference files
        if original:
                depth_diff_fns = h.fn_list(workdir, f'{basin}*wy{WY}*diff*depth_original.nc')
                # Only run for the iSnobal diffs removing fns if 'ua' or 'nwm' in filename
                depth_diff_fns = [h for h in depth_diff_fns if 'ua' not in h and 'nwm' not in h]
        else:
                # This is for the uniform reprojection diffs to match the terrain data grid
                depth_diff_fns = h.fn_list(workdir, f'{basin}*wy{WY}*diff*depth_uniformreproj.nc')
                # Only run for UA and NWM diffs removing fns if 'ua' or 'nwm' in filename
                depth_diff_fns = [h for h in depth_diff_fns if 'ua' in h or 'nwm' in h]
        if not depth_diff_fns:
                print(f'No depth diff files found for {basin} WY {WY}')
                return
        else:
                for depth_diff_fn in depth_diff_fns:
                        if verbose:
                                print(f'{depth_diff_fn}\n')
                        dt = PurePath(depth_diff_fn).stem.split('diff_')[1].split('_')[0]
                        runtype = PurePath(depth_diff_fn).stem.split(f'wy{WY}_')[1].split('_')[0]
                        # Locate terrain file
                        terrain_fn = h.fn_list(script_dir, f'{basin}*_setup/data/{basin}_terrain.nc')[0]
                        if verbose:
                                print(f'{terrain_fn}\n')

                        # Load the terrain and SDD shift datasets
                        if verbose:
                                print('Loading datasets...')
                        ds = xr.open_dataset(terrain_fn, drop_variables=['spatial_ref', 'hs', 'slope'])
                        depth_diff_ds = xr.open_dataset(depth_diff_fn)

                        # combine the two datasets
                        combo_ds = xr.merge([ds, depth_diff_ds])
                        # Derive dem elevation ranges from the terrain dataset using hundreds place rounding and flexible bin number
                        p = None
                        if verbose:
                                print('Binning elevation data...')
                        _, dem_elev_ranges = proc.bin_elev(dem=combo_ds['dem'], basinname=basin,
                                                                plot_on=False, round_on=True, p=p, verbose=verbose)
                        # Extract elevation bins from the ranges
                        # Convert dict values into a flat list
                        elevation_bins = [f[0] for jdx, f in enumerate(dem_elev_ranges.values())]
                        # add the last bin (max elev)
                        elevation_bins.append(list(dem_elev_ranges.values())[-1][-1])
                        if verbose:
                                print(f'{elevation_bins}\n')

                        # Plot radial plots of snow depth differences
                        num_aspect_bins = 8
                        aspect_labels = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
                        if p is not None:
                                outdir = f'/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/depth_diff/radial_{num_aspect_bins}_p{p}'
                        else:
                                outdir = f'/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/depth_diff/radial_{num_aspect_bins}'
                        # If directory doesn't exist, create it using pathlib
                        Path(outdir).mkdir(parents=True, exist_ok=True)
                        if verbose:
                                print(outdir)
                        if p is not None:
                                outname = f'{outdir}/{PurePath(depth_diff_fn).stem.split('_original')[0]}_radial_{num_aspect_bins}_p{p}.png'
                        else:
                                outname = f'{outdir}/{PurePath(depth_diff_fn).stem.split('_original')[0]}_radial_{num_aspect_bins}.png'
                        if verbose:
                                print(f'Plotting radial plot to {outname}\n')
                        title = f'{basin.capitalize()} {runtype} {dt}'
                        cmap = sns.palettes.color_palette(palette, as_cmap=True)

                        psr.plot_radial(combo_ds=combo_ds, combo_var='depth_diff', combo_var_units='m',
                                        elevation_bins=elevation_bins,
                                        cmap=cmap, num_aspect_bins=num_aspect_bins,
                                        aspect_labels=aspect_labels,
                                        elev_fontcolor="k", title=title,
                                        vminnorm=-1, vmaxnorm=1, outname=outname)

In [ ]:
process_radial_plots = True # Already have these processed, skip for now
for basin in basins:
    for wy in [2021, 2022, 2023, 2024]:
        print(basin, wy)
        if process_radial_plots:
            crank_snow_depth_diff_radial(basin, WY=wy, original=True, verbose=False)

In [ ]:
# Adjust for UA and NWM resampled terrain

### Slope

In [ ]:
# Generate radial plots of snow depth differences
def extract_slope(basin, script_dir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts',
                  verbose=True):
    # Locate terrain file
    terrain_fn = h.fn_list(script_dir, f'{basin}*_setup/data/{basin}_terrain.nc')[0]
    if verbose:
        print(f'{terrain_fn}\n')

    # Load the terrain dataset
    if verbose:
        print('Loading terrain...')
    ds = xr.open_dataset(terrain_fn, drop_variables=['spatial_ref', 'hs'])
    return ds
def bin_elev(basin, terrain_ds, verbose=True):
    # Derive dem elevation ranges from the terrain dataset using hundreds place rounding and flexible bin number
    if verbose:
        print('Binning elevation data...')
    _, dem_elev_ranges = proc.bin_elev(dem=terrain_ds['dem'], basinname=basin,
                                       plot_on=False, round_on=True, p=None, verbose=verbose)
    # Extract elevation bins from the ranges
    # Convert dict values into a flat list
    elevation_bins = [f[0] for jdx, f in enumerate(dem_elev_ranges.values())]
    # add the last bin (max elev)
    elevation_bins.append(list(dem_elev_ranges.values())[-1][-1])
    if verbose:
        print(f'{elevation_bins}\n')
    return dem_elev_ranges

In [ ]:
process_slope = True
if process_slope:
    ds_list = [extract_slope(basin) for basin in basins]
    dem_elev_range_list = [bin_elev(basin, terrain_ds) for basin, terrain_ds in zip(basins, ds_list)]

In [ ]:
def calc_mean_var_by_elev(dem_elev_ranges, dem, da, outdir, varname='slope', verbose=True):
    # Loop through each elevation range, noting the mean slope for each range
    stat_var_values = {}
    for elev_range in dem_elev_ranges.values():
        # Extract corresponding canopy density values for this elevation range in the dataset
        da_elev_slice = da.where(dem>=elev_range[0]).where(dem<elev_range[1])
        # Calculate stats for this variable in this elevation range
        mean_var_value = np.nanmean(da_elev_slice.values)
        stdev_var_value = np.nanstd(da_elev_slice.values)
        if verbose:
            print(f'Elevation range: {elev_range[0]} - {elev_range[1]} m')
            print(f'    Mean variable value: {mean_var_value:.1f}')
            print(f'    Standard deviation: {stdev_var_value:.1f}')
        # Store stats in the dict as a tuple, using the elev_range as the key
        stat_var_values[elev_range] = (mean_var_value, stdev_var_value)

    # Convert the dict to a dataframe
    stat_var_df = pd.DataFrame.from_dict(stat_var_values, orient='index', columns=[f'mean_{varname}', f'stdev_{varname}'])

    # Then save it to a csv file
    outname = f'{outdir}/{basin}_{varname}_stats_by_elev.csv'
    if verbose:
        print(outname)
    stat_var_df.to_csv(outname)
    return stat_var_df

stat_slope_dflist = []
stat_aspect_dflist = []
outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/terrain'
if not os.path.exists(outdir):
    os.makedirs(outdir)

for ds, dem_elev_ranges, basin in zip(ds_list, dem_elev_range_list, basins):
    # Calculate slope stats by elevation
    if verbose:
        print(f'Processing {basin}...')
    if verbose:
        print('Calculating slope statistics by elevation...')
    stat_slope_df = calc_mean_var_by_elev(dem_elev_ranges=dem_elev_ranges, dem=ds['dem'], da=ds['slope'], outdir=outdir, varname='slope')

    # Calculate mean aspect by elevation
    if verbose:
        print('Calculating aspect statistics by elevation...')
    stat_aspect_df = calc_mean_var_by_elev(dem_elev_ranges=dem_elev_ranges, dem=ds['dem'], da=ds['aspect'], outdir=outdir, varname='aspect')

    stat_slope_dflist.append(stat_slope_df)
    stat_aspect_dflist.append(stat_aspect_df)

In [ ]:
# Plot mean slope by elevation bin
varname = 'slope'
varname_units = 'degrees'
fig, axa = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, basin, stat_df in zip(axa.flatten(), basins, stat_slope_dflist):
    stat_df.plot(ax=ax, y=f'mean_{varname}', yerr=f'stdev_{varname}', title=f'{basin.capitalize()} Mean {varname.capitalize()} by Elevation Bin',
                       ylabel=f'Mean {varname.capitalize()} ({varname_units})', xlabel='Elevation Bin (m)')
    # copy this for manipulation and annotation
    reindexed_df = copy.deepcopy(stat_df)
    reindexed_df.index = np.arange(len(stat_df.index))
    # annotate mean values in black fontcolor on a white backgound
    for idx, row in reindexed_df.iterrows():
        ax.annotate(f'{row[f"mean_{varname}"]:.1f}', xy=(idx, row[f'mean_{varname}']-2.5),
                    ha='center', va='bottom', color='k', fontsize=10, bbox=dict(facecolor='w', alpha=0.85, edgecolor='none'))
        # Do the same for standard deviation values at the top of the error bars
        ax.annotate(f'( {row[f"stdev_{varname}"]:.1f} )', xy=(idx, row[f'mean_{varname}'] + row[f'stdev_{varname}']),
                    ha='center', va='bottom', color='k', fontsize=8, fontstyle='oblique', bbox=dict(facecolor='w', alpha=0.85, edgecolor='none'))
    # Rotate x-axis labels for legibility
    xticks = np.arange(len(stat_df.index))
    ax.set_xticks(xticks)
    ax.set_xticklabels(stat_df.index, rotation=90)
    # Turn off legend
    ax.legend().set_visible(False)
    ax.grid(linestyle=':')

In [ ]:
# Plot mean aspect by elevation bin -- this isn't helpful as is, need to convert to northness and eastness
varname = 'aspect'
varname_units = 'degrees'
fig, axa = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, basin, stat_df in zip(axa.flatten(), basins, stat_aspect_dflist):
    stat_df.plot(ax=ax, y=f'mean_{varname}', yerr=f'stdev_{varname}', title=f'{basin.capitalize()} Mean {varname.capitalize()} by Elevation Bin',
                       ylabel=f'Mean {varname.capitalize()} ({varname_units})', xlabel='Elevation Bin (m)')
    # Rotate x-axis labels for legibility
    xticks = np.arange(len(stat_df.index))
    ax.set_xticks(xticks)
    ax.set_xticklabels(stat_df.index, rotation=90)
    # Turn off legend
    ax.legend().set_visible(False)
    ax.grid(linestyle=':')
    ax.set_ylim(0, 360)  # Aspect is in degrees, so set y-limits to 0-360

In [ ]:
for ds in ds_list:
    print(ds['slope'].max().values)

In [ ]:
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/diffs'
# define slope categories as a range values with 10 degree increments
slope_categories = np.arange(0, 61, 10)
# Construct a dict of slope ranges spanning the slope categories
# each range will be a tuple of (min_slope, max_slope) for each category
slope_ranges = {f'{slope_categories[i]}_{slope_categories[i+1]}': (int(slope_categories[i]), int(slope_categories[i+1]))
                for i in range(len(slope_categories)-1)}
slope_ranges

In [ ]:
def calc_var_stats_by_slope(slope_ranges, da, slope, basin, varname, outdir, verbose=True):
    # Loop through each range, noting the mean value for each
    stat_var_values = {}
    for slope_range in slope_ranges.values():
        # Extract corresponding variable values for this range in the dataset
        da_slice = da.where(slope>=slope_range[0]).where(slope<slope_range[1])
        # Calculate stats for this variable in this elevation range if the slice is not empty
        if not da_slice.isnull().all():
            mean_var_value = np.nanmean(da_slice.values)
            stdev_var_value = np.nanstd(da_slice.values)
            if verbose:
                print(f'Slope range: {slope_range[0]} - {slope_range[1]} m')
                print(f'    Mean variable value: {mean_var_value:.1f}')
                print(f'    Standard deviation: {stdev_var_value:.1f}')
            # Store stats in the dict as a tuple, using the slope_range as the key
            stat_var_values[slope_range] = (mean_var_value, stdev_var_value)
        else:
            stat_var_values[slope_range] = (np.nan, np.nan)  # If the slice is empty, store NaN values

    # Convert the dict to a dataframe
    stat_var_df = pd.DataFrame.from_dict(stat_var_values, orient='index', columns=[f'mean_{varname}', f'stdev_{varname}'])

    # Then save it to a csv file
    outname = f'{outdir}/{basin}_{varname}_stats_by_slope.csv'
    if verbose:
        print(outname)
    stat_var_df.to_csv(outname)
    return stat_var_df

In [ ]:
def plot_stat_var_df(stat_var_df, ax, vname, varname_units, xlab='Slope', rotate_xticks=False):
    stat_var_df.plot(ax=ax, y=f'mean_{vname}', yerr=f'stdev_{vname}',
                    label=f'{basin.capitalize()} Mean {vname.capitalize()}',
                    xlabel=f'{xlab} Bin (m)')
    # copy this for manipulation and annotation
    reindexed_df = copy.deepcopy(stat_var_df)
    reindexed_df.index = np.arange(len(stat_var_df.index))
    # annotate mean values in black fontcolor on a white backgound
    for idx, row in reindexed_df.iterrows():
        ax.annotate(f'{row[f"mean_{vname}"]:.1f}', xy=(idx, row[f'mean_{vname}']),
                    ha='center', va='bottom', color='k', fontsize=10, bbox=dict(facecolor='w', alpha=0.85, edgecolor='none'))
        # Do the same for standard deviation values at the top of the error bars
        ax.annotate(f'( {row[f"stdev_{vname}"]:.1f} )', xy=(idx, row[f'mean_{vname}'] + row[f'stdev_{vname}']),
                    ha='center', va='bottom', color='k', fontsize=8, fontstyle='oblique', bbox=dict(facecolor='w', alpha=0.85, edgecolor='none'))
    xticks = np.arange(len(stat_var_df.index))
    ax.set_xticks(xticks)
    # Rotate x-axis labels for legibility
    if rotate_xticks:
        ax.set_xticklabels(stat_var_df.index, rotation=90)
    else:
        ax.set_xticklabels(stat_var_df.index)
    # Turn off legend
    ax.legend().set_visible(False)
    ax.grid(linestyle=':')

In [ ]:
sns.set_palette('colorblind')

In [ ]:
# Snow depth differences binned by slope
varname = 'depth_diff'

for terrain_ds, basin in zip(ds_list, basins):
    depth_diff_fns = h.fn_list(workdir, f'{basin}*diff*depth_original.nc')

    # Remove UA and NWM
    depth_diff_fns = [h for h in depth_diff_fns if 'ua' not in h and 'nwm' not in h]

    # Extract the dt from the filenames
    dt_list = [PurePath(fn).stem.split('diff_')[1].split('_')[0] for fn in depth_diff_fns]
    # print(dt_list)

    # Extract the runtype from the filenames
    runtype_list = [PurePath(fn).stem.split('_diff')[0].split('_')[-1] for fn in depth_diff_fns]
    # print(runtype_list)

    # Load the depth difference files
    depth_diff_dslist = [xr.open_dataset(depth_diff_fn, chunks='auto') for depth_diff_fn in depth_diff_fns]

    fig, axa = plt.subplots(int(len(depth_diff_fns)/2), 2,
                            sharex=True, sharey=True,
                            figsize=(12, 4*int(len(depth_diff_fns)/2)))
    # Add the runtype and dt to the attributes for each ds
    for ax, ds, dt, runtype in zip(axa.flatten(), depth_diff_dslist, dt_list, runtype_list):
        ds.attrs['dt'] = dt
        ds.attrs['runtype'] = runtype
        # This will be in the outname csv
        vname = f'{runtype}_{dt}_{varname}'
        varname_units = 'm'
        # Bin the depth difference file by the defined slope categories
        stat_var_df = calc_var_stats_by_slope(slope_ranges=slope_ranges,
                                              da=ds['depth_diff'],
                                              slope=terrain_ds['slope'],
                                              basin=basin, varname=vname,
                                              verbose=False,
                                              outdir=outdir)
        # Plot it up
        plot_stat_var_df(stat_var_df=stat_var_df, ax=ax, vname=vname, varname_units=varname_units, xlab='Slope')
        ax.set_ylim(-1, 1.5)
        ax.set_ylabel('mean depth difference (m)')
        ax.set_title(f'{runtype.capitalize()} {dt}')
        # Add zero line at y=0
        ax.axhline(y=0, color='k', linestyle='--', linewidth=1)
    plt.suptitle(f'{basin.capitalize()} Snow Depth Differences by Slope', fontsize=14, y=1.02)
    plt.tight_layout()
    # Save the figure to file
    outname = f'{outdir}/{basin}_depth_diff_by_slope.png'
    if verbose:
        print(f'Saving figure to {outname}')
    plt.savefig(outname, bbox_inches='tight', dpi=300)

In [ ]:
# and to slightly work backwards, get the proportion of the basin that falls into each slope category
fig, axa = plt.subplots(3, 1, figsize=(5, 3*3), sharex=True, sharey=True)
for ax, terrain_ds, basin in zip(axa.flatten(), ds_list, basins):
    slope_counts = []
    for slope_range in slope_ranges.values():
        # Count the number of pixels in each slope category
        slope_slice = terrain_ds['slope'].where((terrain_ds['slope'] >= slope_range[0]) & (terrain_ds['slope'] < slope_range[1]), drop=True)
        slope_count = slope_slice.count().values
        slope_counts.append(slope_count)
    slope_proportions = np.array(slope_counts) / np.array(slope_counts).sum() * 100  # Convert to percentage
    # Turn this into a dataframe \
    slope_df = pd.DataFrame(slope_proportions, index=list(slope_ranges.keys()), columns=['proportion'])
    print(f'{basin.capitalize()} Slope Proportions:')
    # Save to CSV
    outname = f'{outdir}/{basin}_slope_proportions.csv'
    print(outname)
    slope_df.to_csv(outname)
    # Plot slope proportions
    slope_df.plot(kind='bar', ax=ax, color='skyblue')
    # Annotate the bars with the proportion values
    for idx, value in enumerate(slope_proportions):
        ax.text(idx, value + 1.5, f'{value:.1f}%', ha='center', va='bottom', fontsize=10, bbox=dict(facecolor='w', alpha=1, edgecolor='none'))
    # Turn off legend
    ax.legend().set_visible(False)
    ax.set_title(f'{basin.capitalize()} Slope Proportions')
    ax.set_ylabel('Proportion (%)')
    ax.set_xlabel('Slope Category (degrees)')
    ax.grid(linestyle=':')
    plt.tight_layout()
# Save the figure to file
outname = f'{outdir}/blue_animas_yampa_slope_proportions.png'
if verbose:
    print(f'Saving figure to {outname}')
plt.savefig(outname, bbox_inches='tight', dpi=300)

### Canopy cover

In [ ]:
# Read in canopy cover dataset for each basin
def get_basin_terrain(basin_setupdir, verbose=True):
    # Locate topo.nc file within setupdir
    topo_file = h.fn_list(basin_setupdir, '*/*topo.nc')[0]
    if verbose:
        print(topo_file)
    topo = xr.open_dataset(topo_file)
    # topo

    # Extract the DEM for this basin
    dem = topo['dem']
    # dem.plot.imshow()
    return dem


def load_basin_canopy(basin, veg_dir='/uufs/chpc.utah.edu/common/home/u6058223/veg_data',
                       product='science',
                       verbose=True):
    outname = f'{veg_dir}/{product}_{basin}_reprojmatch.tif'
    if verbose:
        print(outname)
    tcc = np.squeeze(xr.open_dataset(outname))
    # tcc['band_data'].plot.imshow()
    return tcc

def bin_canopy_density(basin,
                       veg_dir='/uufs/chpc.utah.edu/common/home/u6058223/veg_data',
                       product='science',
                       script_dir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts',
                       veg_outdir='/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/canopy_density',
                       verbose=True):
    tcc = load_basin_canopy(basin=basin, veg_dir=veg_dir, product=product, verbose=verbose)
    # Read in DEM for each basin
    basin_setupdir = h.fn_list(script_dir, f'*{basin}*setup')[0]
    dem = get_basin_terrain(basin_setupdir=basin_setupdir)
    # dem.plot.imshow()

    # Derive dem elevation ranges from the terrain dataset using hundreds place rounding and flexible bin number
    if verbose:
        print('Binning elevation data...')
    _, dem_elev_ranges = proc.bin_elev(dem=dem, basinname=basin,
                                        plot_on=False, round_on=True, p=None, verbose=verbose)

    # Loop through each elevation range, noting the mean canopy density for each range
    mean_canopy_densities = {}
    for elev_range in dem_elev_ranges.values():
        print(f'Elevation range: {elev_range[0]} - {elev_range[1]} m')
        # Extract corresponding canopy density values for this elevation range in the dataset
        canopy_density = tcc['band_data'].where(dem>=elev_range[0]).where(dem<elev_range[1])
        # Calculate mean canopy density for this range
        mean_canopy_density = np.nanmean(canopy_density.values)
        if verbose:
            print(f'Mean canopy density: {mean_canopy_density:.1f}%')
        # Store mean density in the dict, using the elev_range as the key
        mean_canopy_densities[elev_range] = mean_canopy_density

    # Convert the dict to a dataframe
    mean_canopy_df = pd.DataFrame.from_dict(mean_canopy_densities, orient='index', columns=['mean_canopy_density'])

    # Then save it to a csv file
    outname = f'{veg_outdir}/{basin}_mean_canopy_density.csv'
    mean_canopy_df.to_csv(outname)
    if verbose:
        print(f'Saving to {outname}')

In [ ]:
process_canopy_density = False # Already have these processed, skip for now
for basin in basins:
    if process_canopy_density:
        bin_canopy_density(basin)

In [ ]:
# Read it all back in!
veg_outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/canopy_density'
veg_list = []

for basin in basins:
    # Read in the canopy density data
    veg_df = pd.read_csv(f'{veg_outdir}/{basin}_mean_canopy_density.csv')
    # Specify column names
    veg_df.columns = [f'{basin}_elevation_range', 'mean_canopy_density']
    # Convert elevation_range from string to tuple
    veg_df[f'{basin}_elevation_range'] = veg_df[f'{basin}_elevation_range'].apply(lambda x: eval(x))
    veg_list.append(veg_df)

In [ ]:
# Plot of canopy density by binned elevations! This addresses veg stuff
fig, axa = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for idx, basin in enumerate(basins):
    ax = axa[idx]
    mean_canopy_densities = veg_list[idx].set_index(f'{basin}_elevation_range')['mean_canopy_density'].to_dict()
    # Plot the mean canopy density for each elevation range
    elev_ranges = list(mean_canopy_densities.keys())
    mean_densities = list(mean_canopy_densities.values())
    ax.plot([f'{elev[0]}-{elev[1]}' for elev in elev_ranges], mean_densities, marker='o')
    ax.set_ylabel('Canopy Density (%)')
    ax.set_title(f'{basin.capitalize()} Canopy Density by Elevation Range')
    # Rotate the xticks for each axis
    ax.set_xticklabels([f'{elev[0]}-{elev[1]}' for elev in elev_ranges], rotation=45, ha='right')

In [ ]:
def calc_var_stats_by_canopy(canopy_ranges, da, canopy, basin, varname, outdir, verbose=True):
    # Loop through each range, noting the mean value for each
    stat_var_values = {}
    for canopy_range in canopy_ranges.values():
        # Extract corresponding variable values for this range in the dataset
        # da_slice = da.where(canopy>=canopy_range[0]).where(canopy<canopy_range[1], drop=True)
        da_slice = da.where((canopy>=canopy_range[0]).values).where((canopy<canopy_range[1]).values)
        # Calculate stats for this variable in this elevation range if the slice is not empty
        if not da_slice.isnull().all():
            mean_var_value = np.nanmean(da_slice.values)
            stdev_var_value = np.nanstd(da_slice.values)
            if verbose:
                print(f'Canopy range: {canopy_range[0]} - {canopy_range[1]} %')
                print(f'    Mean variable value: {mean_var_value:.1f}')
                print(f'    Standard deviation: {stdev_var_value:.1f}')
            # Store stats in the dict as a tuple, using the slope_range as the key
            stat_var_values[canopy_range] = (mean_var_value, stdev_var_value)
        else:
            stat_var_values[canopy_range] = (np.nan, np.nan)  # If the slice is empty, store NaN values

    # Convert the dict to a dataframe
    stat_var_df = pd.DataFrame.from_dict(stat_var_values, orient='index', columns=[f'mean_{varname}', f'stdev_{varname}'])

    # Then save it to a csv file
    outname = f'{outdir}/{basin}_{varname}_stats_by_canopy.csv'
    if verbose:
        print(outname)
    stat_var_df.to_csv(outname)
    return stat_var_df

In [ ]:
veg_dir='/uufs/chpc.utah.edu/common/home/u6058223/veg_data'
product='science'
verbose=True

tcc_list = [load_basin_canopy(basin=basin, veg_dir=veg_dir, product=product, verbose=verbose) for basin in basins]
_ = [print(tcc['band_data'].max().values) for tcc in tcc_list]

In [ ]:
# define canopy categories as a range values with 10 % increments
canopy_categories = np.arange(0, 101, 10)
# Construct a dict of slope ranges spanning the slope categories
# each range will be a tuple of (min_slope, max_slope) for each category
canopy_ranges = {f'{canopy_categories[i]}_{canopy_categories[i+1]}': (int(canopy_categories[i]), int(canopy_categories[i+1]))
                for i in range(len(canopy_categories)-1)}
canopy_ranges

In [ ]:
# Snow depth differences binned canopy density
varname = 'depth_diff'

for tcc, basin, in zip(tcc_list, basins):
    depth_diff_fns = h.fn_list(workdir, f'{basin}*diff*depth_original.nc')

    # Remove UA and NWM
    depth_diff_fns = [h for h in depth_diff_fns if 'ua' not in h and 'nwm' not in h]

    # Extract the dt from the filenames
    dt_list = [PurePath(fn).stem.split('diff_')[1].split('_')[0] for fn in depth_diff_fns]
    # print(dt_list)

    # Extract the runtype from the filenames
    runtype_list = [PurePath(fn).stem.split('_diff')[0].split('_')[-1] for fn in depth_diff_fns]
    # print(runtype_list)

    # Load the depth difference files
    depth_diff_dslist = [xr.open_dataset(depth_diff_fn, chunks='auto') for depth_diff_fn in depth_diff_fns]

    fig, axa = plt.subplots(int(len(depth_diff_fns)/2), 2,
                            sharex=True, sharey=True,
                            figsize=(12, 4*int(len(depth_diff_fns)/2)))
    # Add the runtype and dt to the attributes for each ds
    for ax, ds, dt, runtype in zip(axa.flatten(), depth_diff_dslist, dt_list, runtype_list):
        ds.attrs['dt'] = dt
        ds.attrs['runtype'] = runtype
        # This will be in the outname csv
        vname = f'{runtype}_{dt}_{varname}'
        varname_units = 'm'

        # Bin the depth difference file by the defined canopy density categories
        stat_var_df = calc_var_stats_by_canopy(canopy_ranges=canopy_ranges,
                                            da=ds['depth_diff'],
                                            canopy=tcc['band_data'],
                                            basin=basin, varname=vname,
                                            verbose=False,
                                            outdir=outdir)
        # Plot it up
        plot_stat_var_df(stat_var_df=stat_var_df, ax=ax, vname=vname, varname_units=varname_units, xlab='Canopy Density (%)', rotate_xticks=True)
        ax.set_ylim(-1, 1.5)
        ax.set_ylabel('mean depth difference (m)')
        ax.set_title(f'{runtype.capitalize()} {dt}')
        # Add zero line at y=0
        ax.axhline(y=0, color='k', linestyle='--', linewidth=1)
    plt.suptitle(f'{basin.capitalize()} Snow Depth Differences by Canopy Density', fontsize=14, y=1.02)
    plt.tight_layout()
    # Save the figure to file
    outname = f'{outdir}/{basin}_depth_diff_by_canopy_density.png'
    if verbose:
        print(f'Saving figure to {outname}')
    plt.savefig(outname, bbox_inches='tight', dpi=300)

In [ ]:
# Do the same thing but with SDD, not depth_diff


In [ ]:
# Also to slightly work backwards, get the proportion of the basin that falls into each canopy density bin
fig, axa = plt.subplots(3, 1, figsize=(6, 3*3), sharex=True, sharey=True)
for ax, tcc, basin in zip(axa.flatten(), tcc_list, basins):
    canopy_counts = []
    for canopy_range in canopy_ranges.values():
        # Count the number of pixels in each category
        canopy_slice = tcc['band_data'].where((tcc['band_data'] >= canopy_range[0]) & (tcc['band_data'] < canopy_range[1]), drop=True)
        canopy_count = canopy_slice.count().values
        canopy_counts.append(canopy_count)
    proportions = np.array(canopy_counts) / np.array(canopy_counts).sum() * 100  # Convert to percentage
    # Turn this into a dataframe \
    canopy_df = pd.DataFrame(proportions, index=list(canopy_ranges.keys()), columns=['proportion'])
    print(f'{basin.capitalize()} Canopy Proportions:')
    # Save to CSV
    outname = f'{outdir}/{basin}_canopy_proportions.csv'
    print(outname)
    canopy_df.to_csv(outname)
    # Plot proportions
    canopy_df.plot(kind='bar', ax=ax, color='forestgreen')
    # Annotate the bars with the proportion values
    for idx, value in enumerate(proportions):
        ax.text(idx, value + 0.5, f'{value:.1f}%', ha='center', va='bottom', fontsize=10, bbox=dict(facecolor='w', alpha=1, edgecolor='none'))
    # Turn off legend
    ax.legend().set_visible(False)
    ax.set_title(f'{basin.capitalize()} Canopy Proportions')
    ax.set_ylabel('Basin Proportion (%)')
    ax.set_xlabel('Canopy Density (%)')
    ax.grid(linestyle=':')
    plt.tight_layout()
# Save the figure to file
outname = f'{outdir}/blue_animas_yampa_canopy_proportions.png'
if verbose:
    print(f'Saving figure to {outname}')
plt.savefig(outname, bbox_inches='tight', dpi=300)

In [ ]:
# # Increase fontsize
# plt.rcParams.update({'font.size': 18})

In [ ]:
# Plot it all on one figure
fig, ax = plt.subplots(figsize=(12,8))
for idx, basin in enumerate(basins):
    mean_canopy_densities = veg_list[idx].set_index(f'{basin}_elevation_range')['mean_canopy_density'].to_dict()
    elev_ranges = list(mean_canopy_densities.keys())
    # Calculate the mean value of each elevation range
    mean_elevs = [(elev[0] + elev[1]) / 2 for elev in elev_ranges]
    mean_densities = list(mean_canopy_densities.values())
    # Plot the mean canopy density by mean elevation value of each range
    ax.plot(mean_elevs, mean_densities, markersize=10, marker='o', label=basin.capitalize())
    ax.set_ylabel('Canopy Density (%)')
    ax.set_title('Mean Canopy Density by Mean Elevation Range')
ax.grid(alpha=0.7, linestyle=":")
plt.legend()

In [ ]:
# Plot it way bigger
# Increase fontsize
plt.rcParams.update({'font.size': 20})
fig, ax = plt.subplots(figsize=(8,6))
colors = ['darkslateblue',
          'goldenrod',
          'lightcoral']
markers = ['X', 'o', '|']
markersizes = [12, 12, 20]
for idx, basin in enumerate(basins):
    mean_canopy_densities = veg_list[idx].set_index(f'{basin}_elevation_range')['mean_canopy_density'].to_dict()
    elev_ranges = list(mean_canopy_densities.keys())
    # Calculate the mean value of each elevation range
    mean_elevs = [(elev[0] + elev[1]) / 2 for elev in elev_ranges]
    mean_densities = list(mean_canopy_densities.values())
    # Plot the mean canopy density by mean elevation value of each range
    ax.plot(mean_elevs, mean_densities, markersize=markersizes[idx], linewidth=4, marker=markers[idx], label=basin.capitalize(), color=colors[idx])
    ax.set_xlabel('Mean Elevation (m)')
    ax.set_ylabel('Canopy Density (%)')
    # ax.set_title('Mean Canopy Density by \nMean Elevation Range')
ax.grid(alpha=0.7, linestyle=":", linewidth=1.5)
# Move legend above plot with no box in 1 row and 3 columns
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=3, frameon=False, fontsize=18)
# Move legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, fontsize=18)
plt.tight_layout()